In [130]:
%load_ext dotenv
%dotenv /home/aurora/.env
%matplotlib inline

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [131]:
import numpy as np
import pandas as pd
import datetime as dt
from google.cloud import bigquery
import re
import pymongo
import os
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from flatten_json import flatten
from tqdm import tqdm
cursor = pymongo.MongoClient("mongodb://" + os.environ['user'] + ':' + 
                             os.environ['pass'] + '@' + os.environ['db1']  + "/?authSource=" + 
                             os.environ['dbname'])
#warnings.filterwarnings('ignore')
client = bigquery.Client.from_service_account_json('/home/analytics/.secure_files/hitwicketsuperstars-f3e8c620a88c.json')

In [132]:
start = dt.datetime(2019,6,20)
start1 = str(start)
end = dt.datetime(2019,6,26)
end1 = str(end)
print(start,end)

2019-06-20 00:00:00 2019-06-26 00:00:00


In [133]:
app_version = '2.0.2'

# new user reference

In [134]:
client = bigquery.Client.from_service_account_json('/home/analytics/.secure_files/hitwicketsuperstars-f3e8c620a88c.json')
query = (
    f"""SELECT
    * FROM 
    `hitwicketsuperstars.analytics_190927423.new_user_reference`"""
)
new_user_reference = client.query(query).to_dataframe()

In [135]:
new_user_reference['user_first_touch_timestamp'] = new_user_reference['user_first_touch_timestamp'].astype('datetime64[s]')
android_new = new_user_reference[(new_user_reference['user_first_touch_timestamp'] >= start)]

# ftue completed users

In [136]:
query = (f"""
             SELECT 
             user_id as device_id,
             event_timestamp
             FROM `hitwicketsuperstars.analytics_190927423.events_*`,
             UNNEST(event_params) AS params    
             WHERE _TABLE_SUFFIX BETWEEN "{start1.replace('-','')}"
             AND "{end1.replace('-','')}"
             AND app_info.version = '{app_version}'
            AND device.operating_system = 'ANDROID'
             AND params.key = 'action'
            AND event_name IN ('natasha')
           AND params.value.string_value = 'hand_pointer_achievements_clicked'""")
         
ftue_complete = client.query(query).to_dataframe() 

In [137]:
ftue_complete.columns = ['device_id','create_time']
ftue_complete.sort_values('create_time',inplace=True,ascending=False)
ftue_complete.drop_duplicates('device_id',inplace=True)
ftue_complete = ftue_complete[ftue_complete['device_id'].isin(android_new['device_id'])]
ftue_complete['create_time'] = pd.to_datetime(ftue_complete['create_time'], unit = 'us')
print(len(ftue_complete))
ftue_complete.head()

673


,device_id,create_time
623,602f46bae4430f13fa5f1d7d18a70644,2019-06-26 18:09:09.186007
606,efcced8f07408ab9b357213e1bad9d6c,2019-06-26 17:17:31.192007
637,9fd654fc07acf3187139049d032a5b31,2019-06-26 17:16:45.178008
590,08aeabee748b7666d24a831b7718b670,2019-06-26 17:03:33.647008
617,3bc49bdf96d184d85d15f08b5e22f1b3,2019-06-26 16:59:10.874007


In [138]:
c_users = cursor.superstars.users
aw_users = []
for documents in c_users.find({'created_at': {'$gte': start}},{"sign_up_details",'login_details.last_request_at'}): # end condition
    aw_users.append(documents)
dic_flattened = [flatten(d) for d in aw_users]
users = pd.DataFrame(dic_flattened)
users = users[["_id","sign_up_details_device_id",'login_details_last_request_at']]
users.columns = ["user_id","device_id",'last_request']
len(users)

2257

In [139]:
users.sort_values(['device_id','last_request'],ascending=False,inplace=True)
users.drop_duplicates('device_id',inplace=True)
print(len(users))

2196


In [140]:
ftue_complete_user = pd.merge(ftue_complete,users,on='device_id')
ftue_complete_user = ftue_complete_user[['user_id','device_id','create_time','last_request']]

In [141]:
ftue_complete_user.head()

,user_id,device_id,create_time,last_request
0,5d13b4587119e60013b3e9d6,602f46bae4430f13fa5f1d7d18a70644,2019-06-26 18:09:09.186007,2019-06-26 18:12:57.728
1,5d1249a7c5ebf8001185851f,efcced8f07408ab9b357213e1bad9d6c,2019-06-26 17:17:31.192007,2019-06-26 17:19:00.137
2,5d13a7f8114e23001a9596e9,9fd654fc07acf3187139049d032a5b31,2019-06-26 17:16:45.178008,2019-06-26 17:16:55.670
3,5d13a4b4114e23001a954a35,08aeabee748b7666d24a831b7718b670,2019-06-26 17:03:33.647008,2019-06-26 17:06:47.956
4,5d1112c4fa80e30028d2b431,3bc49bdf96d184d85d15f08b5e22f1b3,2019-06-26 16:59:10.874007,2019-06-26 17:13:01.970


In [142]:
ftue_complete_user['d1'] = (ftue_complete_user['last_request']-ftue_complete_user['create_time'])>'24:00:00'

In [143]:
print(len(ftue_complete_user))
ftue_complete_user.head()

673


,user_id,device_id,create_time,last_request,d1
0,5d13b4587119e60013b3e9d6,602f46bae4430f13fa5f1d7d18a70644,2019-06-26 18:09:09.186007,2019-06-26 18:12:57.728,False
1,5d1249a7c5ebf8001185851f,efcced8f07408ab9b357213e1bad9d6c,2019-06-26 17:17:31.192007,2019-06-26 17:19:00.137,False
2,5d13a7f8114e23001a9596e9,9fd654fc07acf3187139049d032a5b31,2019-06-26 17:16:45.178008,2019-06-26 17:16:55.670,False
3,5d13a4b4114e23001a954a35,08aeabee748b7666d24a831b7718b670,2019-06-26 17:03:33.647008,2019-06-26 17:06:47.956,False
4,5d1112c4fa80e30028d2b431,3bc49bdf96d184d85d15f08b5e22f1b3,2019-06-26 16:59:10.874007,2019-06-26 17:13:01.970,False


# users buying TC2

In [144]:
c_end_training = cursor.superstars.user_collectables_logs
aw_end_training_coins = []
for documents in c_end_training.aggregate([{'$unwind':"$data"},   # used for ending training once the training has started
                    {"$match" : {"data.reason_type" : 'TC2_PURCHASE', # shown as 'FINISH' right next to 'SPEEDUP', once the training starts
                                  "type":"HARD_CURRENCY",               
                                  'data.quantity': {'$lt': 0},           
                                  'data.created_at': {'$gte': start}}}]):
    aw_end_training_coins.append(documents)
    
dic_flattened = [flatten(d) for d in aw_end_training_coins]
tc2 = pd.DataFrame(dic_flattened)
tc2 = tc2[["_id","data_created_at",'user']]
tc2.columns = ["tc2_id", "tc2_purchased_at","user_id"]

In [145]:
tc2.sort_values(['user_id','tc2_purchased_at'],ascending=False,inplace=True)
tc2.drop_duplicates('user_id',inplace=True)
print(len(tc2))
tc2.head()

155


,tc2_id,tc2_purchased_at,user_id
173,5d14906c114e23001ac33932,2019-06-27 09:46:20.538,5d148f3c114e23001ac30bf1
172,5d148dd8114e23001ac2eca9,2019-06-27 09:35:51.287,5d148d13114e23001ac2dce9
170,5d147a707119e60013d79968,2019-06-27 08:12:32.132,5d1478e9114e23001abebecb
169,5d14444a7119e60013cb8f76,2019-06-27 08:38:54.360,5d144300114e23001ab07e28
168,5d14155e114e23001aa56dd6,2019-06-27 01:01:18.148,5d1414b9114e23001aa559ff


In [146]:
tc2_user = pd.merge(ftue_complete_user,tc2,on='user_id')
tc2_user = tc2_user[tc2_user['tc2_purchased_at']-tc2_user['create_time']<'24:00:00']
len(tc2_user)

59

In [147]:
grouped = tc2_user.groupby('user_id').agg({'tc2_id':'count'}).reset_index()
print(len(grouped))
grouped.head()

59


,user_id,tc2_id
0,5d0dab60fa80e3002833c145,1
1,5d0dd0defa80e300283dfa70,1
2,5d0de49dfa80e3002842fc90,1
3,5d0de671fa80e30028437034,1
4,5d0e5e208185c200194e76da,1


# planning and grouping

In [148]:
grouped_zero = pd.merge(grouped,ftue_complete_user[['user_id','d1']],on='user_id',how='right')
grouped_zero = grouped_zero.fillna(0)
len(grouped_zero)

673

In [149]:
grouped_zero.head()

,user_id,tc2_id,d1
0,5d0dab60fa80e3002833c145,1.0,False
1,5d0dd0defa80e300283dfa70,1.0,True
2,5d0de49dfa80e3002842fc90,1.0,False
3,5d0de671fa80e30028437034,1.0,True
4,5d0e5e208185c200194e76da,1.0,False


In [150]:
distribution = grouped_zero.groupby('tc2_id').agg({'user_id':'count','d1':'sum'})

In [151]:
distribution['d1%'] = distribution['d1']/distribution['user_id']*100

In [152]:
distribution

,user_id,d1,d1%
tc2_id,,,
0.0,614,112.0,18.241042
1.0,59,14.0,23.728814
